In [1]:
#| default_exp core.tokenization
#| export

from collections import Counter
from typing import Dict, List, Optional, Set, Tuple

import numpy as np

# Constants for memory calculations
KB_TO_BYTES = 1024  # Kilobytes to bytes conversion

In [2]:
#| export
class Tokenizer:
    """
    Base tokenizer class providing the interface for all tokenizers.

    This defines the contract that all tokenizers must follow:
    - encode(): text → list of token IDs
    - decode(): list of token IDs → text
    """

    def encode(self, text: str) -> List[int]:
        """
        Convert text to a list of token IDs.

        TODO: Implement encoding logic in subclasses

        APPROACH:
        1. Subclasses will override this method
        2. Return list of integer token IDs

        EXAMPLE:
        >>> tokenizer = CharTokenizer(['a', 'b', 'c'])
        >>> tokenizer.encode("abc")
        [0, 1, 2]
        """
        ### BEGIN SOLUTION
        raise NotImplementedError(
            f"encode() not implemented in base Tokenizer class\n"
            f"  ❌ Called encode() on abstract base class {self.__class__.__name__}\n"
            f"  💡 Tokenizer is an interface - use a concrete implementation like CharTokenizer or BPETokenizer\n"
            f"  🔧 Example: tokenizer = CharTokenizer(['a', 'b', 'c']); tokenizer.encode('abc')"
        )
        ### END SOLUTION

    def decode(self, tokens: List[int]) -> str:
        """
        Convert list of token IDs back to text.

        TODO: Implement decoding logic in subclasses

        APPROACH:
        1. Subclasses will override this method
        2. Return reconstructed text string

        EXAMPLE:
        >>> tokenizer = CharTokenizer(['a', 'b', 'c'])
        >>> tokenizer.decode([0, 1, 2])
        "abc"
        """
        ### BEGIN SOLUTION
        raise NotImplementedError(
            f"decode() not implemented in base Tokenizer class\n"
            f"  ❌ Called decode() on abstract base class {self.__class__.__name__}\n"
            f"  💡 Tokenizer is an interface - use a concrete implementation like CharTokenizer or BPETokenizer\n"
            f"  🔧 Example: tokenizer = CharTokenizer(['a', 'b', 'c']); tokenizer.decode([0, 1, 2])"
        )
        ### END SOLUTION

In [3]:
def test_unit_base_tokenizer():
    """🧪 Test base tokenizer interface."""
    print("🧪 Unit Test: Base Tokenizer Interface...")

    # Test that base class defines the interface
    tokenizer = Tokenizer()

    # Should raise NotImplementedError for both methods
    try:
        tokenizer.encode("test")
        assert False, "encode() should raise NotImplementedError"
    except NotImplementedError:
        pass

    try:
        tokenizer.decode([1, 2, 3])
        assert False, "decode() should raise NotImplementedError"
    except NotImplementedError:
        pass

    print("✅ Base tokenizer interface works correctly!")

if __name__ == "__main__":
    test_unit_base_tokenizer()

🧪 Unit Test: Base Tokenizer Interface...
✅ Base tokenizer interface works correctly!


In [4]:
#| export
class CharTokenizer(Tokenizer):
    """
    Character-level tokenizer that treats each character as a separate token.

    This is the simplest tokenization approach - every character in the
    vocabulary gets its own unique ID.
    """

    def __init__(self, vocab: Optional[List[str]] = None):
        """
        Initialize character tokenizer.

        TODO: Set up vocabulary mappings

        APPROACH:
        1. Store vocabulary list
        2. Create char→id and id→char mappings
        3. Handle special tokens (unknown character)

        EXAMPLE:
        >>> tokenizer = CharTokenizer(['a', 'b', 'c'])
        >>> tokenizer.vocab_size
        4  # 3 chars + 1 unknown token
        """
        ### BEGIN SOLUTION
        if vocab is None:
            vocab = []

        # add special unknown token
        self.vocab = ['<UNK>'] + vocab
        self.vocab_size = len(self.vocab)

        # create bidirectional mappings
        self.char_to_id = {char: idx for idx, char in enumerate(self.vocab)}
        self.id_to_char = {idx: char for idx, char in enumerate(self.vocab)}

        # store unknown token ID
        self.unk_id = 0
        ### END SOLUTION

    def build_vocab(self, corpus: List[str]) -> None:
        """
        Build vocabulary from a corpus of text.

        TODO: Extract unique characters and build vocabulary

        APPROACH:
        1. Collect all unique characters from corpus
        2. Sort for consistent ordering
        3. Rebuild mappings with new vocabulary

        HINTS:
        - Use set() to find unique characters
        - Join all texts then convert to set
        - Don't forget the <UNK> token
        """
        ### BEGIN SOLUTION
        # Collect all unique characters
        all_chars = set()
        for text in corpus:
            all_chars.update(text)

        # sort for consistent ordering
        unique_chars = sorted(all_chars)

        # rebuild vocabulary with <UNK> token first
        self.vocab = ['<UNK>'] + unique_chars
        self.vocab_size = len(self.vocab)

        # rebuild mappings
        self.char_to_id = {char: idx for idx, char in enumerate(self.vocab)}
        self.id_to_char = {idx: char for idx, char in enumerate(self.vocab)}

        ### END SOLUTION

    def encode(self, text: str) -> List[int]:
        """
        Encode text to list of character IDs.

        TODO: Convert each character to its vocabulary ID

        APPROACH:
        1. Iterate through each character in text
        2. Look up character ID in vocabulary
        3. Use unknown token ID for unseen characters

        EXAMPLE:
        >>> tokenizer = CharTokenizer(['h', 'e', 'l', 'o'])
        >>> tokenizer.encode("hello")
        [1, 2, 3, 3, 4]  # maps to h,e,l,l,o
        """
        ### BEGIN SOLUTION
        tokens = []
        for char in text:
            tokens.append(self.char_to_id.get(char, self.unk_id))
        return tokens
        ### END SOLUTION

    def decode(self, tokens: List[int]) -> str:
        """
        Decode list of token IDs back to text.

        TODO: Convert each token ID back to its character

        APPROACH:
        1. Look up each token ID in vocabulary
        2. Join characters into string
        3. Handle invalid token IDs gracefully

        EXAMPLE:
        >>> tokenizer = CharTokenizer(['h', 'e', 'l', 'o'])
        >>> tokenizer.decode([1, 2, 3, 3, 4])
        "hello"
        """
        ### BEGIN SOLUTION
        chars = []
        for token_id in tokens:
            char = self.id_to_char.get(token_id, '<UNK>')
            chars.append(char)
        return ''.join(chars)
        ### END SOLUTION

In [5]:
def test_unit_char_tokenizer():
    """🧪 Test character tokenizer implementation."""
    print("🧪 Unit Test: Character Tokenizer...")

    # Test basic functionality
    vocab = ['h', 'e', 'l', 'o', ' ', 'w', 'r', 'd']
    tokenizer = CharTokenizer(vocab)

    # Test vocabulary setup
    assert tokenizer.vocab_size == 9  # 8 chars + UNK
    assert tokenizer.vocab[0] == '<UNK>'
    assert 'h' in tokenizer.char_to_id

    # Test encoding
    text = "hello"
    tokens = tokenizer.encode(text)
    expected = [1, 2, 3, 3, 4]  # h,e,l,l,o (based on actual vocab order)
    assert tokens == expected, f"Expected {expected}, got {tokens}"

    # Test decoding
    decoded = tokenizer.decode(tokens)
    assert decoded == text, f"Expected '{text}', got '{decoded}'"

    # Test unknown character handling
    tokens_with_unk = tokenizer.encode("hello!")
    assert tokens_with_unk[-1] == 0  # '!' should map to <UNK>

    # Test vocabulary building
    corpus = ["hello world", "test text"]
    tokenizer.build_vocab(corpus)
    assert 't' in tokenizer.char_to_id
    assert 'x' in tokenizer.char_to_id

    print("✅ Character tokenizer works correctly!")

if __name__ == "__main__":
    test_unit_char_tokenizer()

🧪 Unit Test: Character Tokenizer...
✅ Character tokenizer works correctly!


In [6]:
#| export
def _count_byte_pairs(word_tokens: Dict[str, List[str]], word_freq: Counter) -> Counter:
    """
    Count frequency of all adjacent token pairs across all words.

    Each pair's count is weighted by how often its containing word appears
    in the corpus, so frequent words contribute more to pair statistics.

    TODO: Count all adjacent pairs weighted by word frequency

    APPROACH:
    1. Iterate through each word and its frequency
    2. Get all adjacent pairs from the word's tokens
    3. Add the word's frequency to each pair's count
    4. Return the Counter of pair frequencies

    EXAMPLE:
    >>> word_tokens = {"hello": ['h', 'e', 'l', 'l', 'o</w>']}
    >>> word_freq = Counter({"hello": 3})
    >>> counts = _count_byte_pairs(word_tokens, word_freq)
    >>> counts[('h', 'e')]
    3

    HINT: For each word, get pairs with a zip-based loop, then add freq to each pair count
    """
    ### BEGIN SOLUTION
    pair_counts = Counter()

    for word, freq in word_freq.items():
        tokens = word_tokens[word]

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i+1])
            pair_counts[pair] += freq
    return pair_counts
    ### END SOLUTION

In [7]:
def test_unit_count_byte_pairs():
    """🧪 Test byte pair counting with frequency weighting."""
    print("🧪 Unit Test: Count Byte Pairs...")

    # Two words: "hello" appears 3 times, "help" appears 1 time
    word_tokens = {
        "hello": ['h', 'e', 'l', 'l', 'o</w>'],
        "help": ['h', 'e', 'l', 'p</w>']
    }
    word_freq = Counter({"hello": 3, "help": 1})

    counts = _count_byte_pairs(word_tokens, word_freq)

    # ('h','e') appears in both words: 3 + 1 = 4
    assert counts[('h', 'e')] == 4, f"Expected 4, got {counts[('h', 'e')]}"

    # ('e','l') appears in both words: 3 + 1 = 4
    assert counts[('e', 'l')] == 4, f"Expected 4, got {counts[('e', 'l')]}"

    # ('l','l') appears only in "hello" (freq 3)
    assert counts[('l', 'l')] == 3, f"Expected 3, got {counts[('l', 'l')]}"

    # ('l','p</w>') appears only in "help" (freq 1)
    assert counts[('l', 'p</w>')] == 1, f"Expected 1, got {counts[('l', 'p</w>')]}"

    # Empty case
    empty_counts = _count_byte_pairs({}, Counter())
    assert len(empty_counts) == 0

    print("✅ Byte pair counting works correctly!")

if __name__ == "__main__":
    test_unit_count_byte_pairs()

🧪 Unit Test: Count Byte Pairs...
✅ Byte pair counting works correctly!


In [8]:
#| export
def _merge_pair(word_tokens: Dict[str, List[str]], pair: Tuple[str, str]) -> str:
    """
    Merge the most frequent pair in all word token lists.

    Scans through every word's tokens and replaces adjacent occurrences
    of the pair with a single concatenated token. Modifies word_tokens
    in place and returns the new merged token string.

    TODO: Merge the given pair in all word token sequences

    APPROACH:
    1. For each word in word_tokens, scan through its token list
    2. When two adjacent tokens match the pair, replace with concatenation
    3. Otherwise keep the token as-is
    4. Update word_tokens in place, return the merged token string

    EXAMPLE:
    >>> word_tokens = {"hello": ['h', 'e', 'l', 'l', 'o</w>']}
    >>> merged = _merge_pair(word_tokens, ('h', 'e'))
    >>> word_tokens["hello"]
    ['he', 'l', 'l', 'o</w>']
    >>> merged
    'he'

    HINTS:
    - Use a while loop with index i to scan each word's tokens
    - When pair matches at position i, append pair[0]+pair[1] and skip 2
    - Otherwise append tokens[i] and advance by 1
    """
    ### BEGIN SOLUTION
    merged_token = pair[0] + pair[1]

    for word in word_tokens:
        tokens = word_tokens[word]
        new_tokens = []
        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1 and
                tokens[i] == pair[0] and
                tokens[i + 1] == pair[1]):
                # merge pair
                new_tokens.append(merged_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        word_tokens[word] = new_tokens

    return merged_token
    ### END SOLUTION

In [9]:
def test_unit_merge_pair():
    """🧪 Test byte pair merging across word token lists."""
    print("🧪 Unit Test: Merge Pair...")

    # Set up word tokens
    word_tokens = {
        "hello": ['h', 'e', 'l', 'l', 'o</w>'],
        "help": ['h', 'e', 'l', 'p</w>']
    }

    # Merge ('h', 'e') → 'he'
    merged = _merge_pair(word_tokens, ('h', 'e'))
    assert merged == 'he', f"Expected 'he', got '{merged}'"
    assert word_tokens["hello"] == ['he', 'l', 'l', 'o</w>'], \
        f"Expected ['he', 'l', 'l', 'o</w>'], got {word_tokens['hello']}"
    assert word_tokens["help"] == ['he', 'l', 'p</w>'], \
        f"Expected ['he', 'l', 'p</w>'], got {word_tokens['help']}"

    # Now merge ('l', 'l') → 'll' (only affects "hello")
    merged2 = _merge_pair(word_tokens, ('l', 'l'))
    assert merged2 == 'll', f"Expected 'll', got '{merged2}'"
    assert word_tokens["hello"] == ['he', 'll', 'o</w>'], \
        f"Expected ['he', 'll', 'o</w>'], got {word_tokens['hello']}"
    # "help" unchanged (no 'l','l' pair)
    assert word_tokens["help"] == ['he', 'l', 'p</w>'], \
        f"help should be unchanged, got {word_tokens['help']}"

    # Edge case: pair not present
    word_tokens_empty = {"ab": ['a', 'b</w>']}
    _merge_pair(word_tokens_empty, ('x', 'y'))
    assert word_tokens_empty["ab"] == ['a', 'b</w>'], "No-match merge should leave tokens unchanged"

    print("✅ Byte pair merging works correctly!")

if __name__ == "__main__":
    test_unit_merge_pair()

🧪 Unit Test: Merge Pair...
✅ Byte pair merging works correctly!


In [14]:
#| export
class BPETokenizer(Tokenizer):
    """
    Byte Pair Encoding (BPE) tokenizer that learns subword units.

    BPE works by:
    1. Starting with character-level vocabulary
    2. Finding most frequent character pairs
    3. Merging frequent pairs into single tokens
    4. Repeating until desired vocabulary size
    """

    def __init__(self, vocab_size: int = 1000):
        """
        Initialize BPE tokenizer.

        TODO: Set up basic tokenizer state

        APPROACH:
        1. Store target vocabulary size
        2. Initialize empty vocabulary and merge rules
        3. Set up mappings for encoding/decoding

        EXAMPLE:
        >>> tokenizer = BPETokenizer(vocab_size=1000)
        >>> tokenizer.vocab_size
        1000

        HINT: Initialize vocab and merges as empty lists, mappings as empty dicts
        """
        ### BEGIN SOLUTION
        self.vocab_size = vocab_size
        self.vocab = []
        self.merges = [] # list of (pair, new_token) merges
        self.token_to_id = {}
        self.id_to_token = {}
        ### END SOLUTION

    def _get_word_tokens(self, word: str) -> List[str]:
        """
        Convert word to list of characters with end-of-word marker.

        TODO: Tokenize word into character sequence

        APPROACH:
        1. Split word into characters
        2. Add </w> marker to last character
        3. Return list of tokens

        EXAMPLE:
        >>> tokenizer._get_word_tokens("hello")
        ['h', 'e', 'l', 'l', 'o</w>']

        HINT: Use list() to split word into characters, then modify the last element
        """
        ### BEGIN SOLUTION
        if not word:
            return []
        
        tokens = list(word)
        tokens[-1] += '</w>' # mark end of word
        return tokens
        ### END SOLUTION

    def _get_pairs(self, word_tokens: List[str]) -> Set[Tuple[str, str]]:
        """
        Get all adjacent pairs from word tokens.

        TODO: Extract all consecutive character pairs

        APPROACH:
        1. Iterate through adjacent tokens
        2. Create pairs of consecutive tokens
        3. Return set of unique pairs

        EXAMPLE:
        >>> tokenizer._get_pairs(['h', 'e', 'l', 'l', 'o</w>'])
        {('h', 'e'), ('e', 'l'), ('l', 'l'), ('l', 'o</w>')}

        HINT: Loop from 0 to len(word_tokens)-1 and create tuple pairs
        """
        ### BEGIN SOLUTION
        pairs = set()
        for i in range(len(word_tokens) - 1):
            pairs.add((word_tokens[i], word_tokens[i+1]))
        return pairs
        ### END SOLUTION

    def train(self, corpus: List[str], vocab_size: int = None) -> None:
        """
        Train BPE on corpus to learn merge rules.

        This is the composition function: it initializes character vocabulary,
        then runs a greedy merge loop using _count_byte_pairs() to find the
        best pair and _merge_pair() to apply it.

        TODO: Implement BPE training using the greedy merge loop

        APPROACH:
        1. Build initial character vocabulary from corpus words
        2. Loop: count pairs, find best, merge it, add to vocab
        3. Stop when vocab reaches target size or no pairs remain
        4. Build final mappings

        EXAMPLE:
        >>> corpus = ["hello", "hello", "help"]
        >>> tokenizer = BPETokenizer(vocab_size=20)
        >>> tokenizer.train(corpus)
        >>> len(tokenizer.vocab) <= 20
        True

        HINTS:
        - Use _get_word_tokens() for initial character tokenization
        - Use _count_byte_pairs(word_tokens, word_freq) to find pair frequencies
        - Use _merge_pair(word_tokens, best_pair) to apply the merge
        - Don't forget to call _build_mappings() at the end
        """
        ### BEGIN SOLUTION
        if vocab_size:
            self.vocab_size = vocab_size

        # count words frequencies and initialize character vocabulary
        word_freq = Counter(corpus)
        vocab = set()
        word_tokens = {}

        for word in word_freq:
            tokens = self._get_word_tokens(word)
            word_tokens[word] = tokens
            vocab.update(tokens)

        self.vocab = sorted(vocab)
        if '<UNK>' not in vocab:
            self.vocab = ['<UNK>'] + self.vocab

        # greedy merge loop: count pairs, merge best, repeat
        self.merges = []

        while len(self.vocab) < self.vocab_size:
            pair_counts = _count_byte_pairs(word_tokens, word_freq)
            if not pair_counts:
                break

            best_pair = pair_counts.most_common(1)[0][0]
            merged_token = _merge_pair(word_tokens, best_pair)
            self.vocab.append(merged_token)
            self.merges.append(best_pair)

        self._build_mappings()
        ### END SOLUTION

    def _build_mappings(self):
        """Build token-to-ID and ID-to-token mappings."""
        ### BEGIN SOLUTION
        self.token_to_id = {token: idx for idx, token in enumerate(self.vocab)}
        self.id_to_token = {idx: token for idx, token in enumerate(self.vocab)}

        ### END SOLUTION

    def _apply_merges(self, tokens: List[str]) -> List[str]:
        """
        Apply learned merge rules to token sequence.

        TODO: Apply BPE merges to token list

        APPROACH:
        1. Start with character-level tokens
        2. Apply each merge rule in order
        3. Continue until no more merges possible

        EXAMPLE:
        >>> # After training, merges might be [('h','e'), ('l','l')]
        >>> tokenizer._apply_merges(['h','e','l','l','o</w>'])
        ['he','ll','o</w>']  # Applied both merges

        HINT: For each merge pair, scan through tokens and replace adjacent pairs
        """
        ### BEGIN SOLUTION
        if not self.merges:
            return tokens
        
        for merge_pair in self.merges:
            new_tokens = []
            i = 0
            while i < len(tokens):
                if (i < len(tokens) - 1 and
                    tokens[i] == merge_pair[0] and
                    tokens[i + 1] == merge_pair[1]):

                    # apply merge
                    new_tokens.append(merge_pair[0] + merge_pair[1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return tokens
        ### END SOLUTION

    def encode(self, text: str) -> List[int]:
        """
        Encode text using BPE.

        TODO: Apply BPE encoding to text

        APPROACH:
        1. Split text into words
        2. Convert each word to character tokens
        3. Apply BPE merges
        4. Convert to token IDs

        EXAMPLE:
        >>> tokenizer.encode("hello world")
        [12, 45, 78]  # Token IDs after BPE merging

        HINTS:
        - Use text.split() for simple word splitting
        - Use _get_word_tokens() to get character-level tokens for each word
        - Use _apply_merges() to apply learned merge rules
        - Use token_to_id dictionary with 0 (UNK) as default
        """
        ### BEGIN SOLUTION
        if not self.vocab:
            return []
        
        # simple word splitting (could be more sophisticated)
        words = text.split()
        all_tokens = []

        for word in words:
            # get character-level tokens
            word_tokens = self._get_word_tokens(word)

            # apply bpe merges
            merged_tokens = self._apply_merges(word_tokens)

            all_tokens.extend(merged_tokens)

        # convert to IDs
        token_ids = []
        for token in all_tokens:
            token_ids.append(self.token_to_id.get(token, 0)) # 0 = <UNK>

        return token_ids
        ### END SOLUTION

    def decode(self, tokens: List[int]) -> str:
        """
        Decode token IDs back to text.

        TODO: Convert token IDs back to readable text

        APPROACH:
        1. Convert IDs to tokens
        2. Join tokens together
        3. Clean up word boundaries and markers

        EXAMPLE:
        >>> tokenizer.decode([12, 45, 78])
        "hello world"  # Reconstructed text

        HINTS:
        - Use id_to_token dictionary with '<UNK>' as default
        - Join all tokens into single string with ''.join()
        - Replace '</w>' markers with spaces for word boundaries
        """
        ### BEGIN SOLUTION
        if not self.id_to_token:
            return ""
        
        # convert IDs to tokens
        token_strings = []
        for token_id in tokens:
            token = self.id_to_token.get(token_id, '<UNK>')
            token_strings.append(token)

        # join and clean up
        text = ''.join(token_strings)

        # replace end-of-word markers with spaces
        text = text.replace('</w>', ' ')

        # clean up extra spaces
        text = ' '.join(text.split())
        return text
        ### END SOLUTION

In [15]:
def test_unit_bpe_tokenizer():
    """🧪 Test BPE tokenizer implementation."""
    print("🧪 Unit Test: BPE Tokenizer...")

    # Test basic functionality with simple corpus
    corpus = ["hello", "world", "hello", "hell"]  # "hell" and "hello" share prefix
    tokenizer = BPETokenizer(vocab_size=20)
    tokenizer.train(corpus)

    # Check that vocabulary was built
    assert len(tokenizer.vocab) > 0
    assert '<UNK>' in tokenizer.vocab

    # Test helper functions
    word_tokens = tokenizer._get_word_tokens("test")
    assert word_tokens[-1].endswith('</w>'), "Should have end-of-word marker"

    pairs = tokenizer._get_pairs(['h', 'e', 'l', 'l', 'o</w>'])
    assert ('h', 'e') in pairs
    assert ('l', 'l') in pairs

    # Test encoding/decoding
    text = "hello"
    tokens = tokenizer.encode(text)
    assert isinstance(tokens, list)
    assert all(isinstance(t, int) for t in tokens)

    decoded = tokenizer.decode(tokens)
    assert isinstance(decoded, str)

    # Test round-trip on training data should work well
    for word in corpus:
        tokens = tokenizer.encode(word)
        decoded = tokenizer.decode(tokens)
        # Allow some flexibility due to BPE merging
        assert len(decoded.strip()) > 0

    print("✅ BPE tokenizer works correctly!")

if __name__ == "__main__":
    test_unit_bpe_tokenizer()

🧪 Unit Test: BPE Tokenizer...
✅ BPE tokenizer works correctly!
